# 06 · Results summary and comparison with IncidentLens

This notebook collects the saved results of notebooks 03–05 into one table (it re-trains nothing).

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*(mismatched devices|pandas version 4).*")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "reports" / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

from rcalens.evaluate import evaluate
base = json.load(open(RESULTS / "baselines.json"))
ml = json.load(open(RESULTS / "ml_ranker.json"))
dl = json.load(open(RESULTS / "deep_learning.json")) if (RESULTS / "deep_learning.json").exists() else None
rows = {"random (expected)": base["overall"]["random (expected)"],
        f"best rule: {base['best_rule']}": base["overall"][base["best_rule"]],
        "XGBoost ranker (grouped CV, mean of 3)": ml["xgb_grouped_cv_mean"],
        "XGBoost ranker, unseen system": ml["xgb_leave_one_system_out"]["all"],
        "XGBoost ranker, unseen culprit service": ml["xgb_leave_one_rc_service_out"]["all"]}
for src, r in ml.get("xgb_loso_by_source", {}).items():
    rows[f"XGBoost, unseen system, {src}"] = r["all"]
if dl:
    rows["1D CNN (grouped CV)"] = dl["cnn_grouped_cv"]["all"]
    rows["XGBoost + CNN ensemble"] = dl["ensemble_xgb_cnn"]["all"]
    rows["1D CNN, unseen system"] = dl["cnn_leave_one_system_out"]["all"]
summary = pd.DataFrame(rows).T[["AC@1", "AC@3", "AC@5", "Avg@5", "MRR"]]
print("trace features:", ml["has_traces"], "| training device:", ml["device"])
summary.round(3)

trace features: True | training device: cuda


,AC@1,AC@3,AC@5,Avg@5,MRR
random (expected),0.055,0.165,0.275,0.165,0.184
best rule: rank average (all features),0.707,0.926,0.963,0.882,0.820
"XGBoost ranker (grouped CV, mean of 3)",0.944,0.990,0.999,0.982,0.968
"XGBoost ranker, unseen system",0.744,0.959,0.982,0.909,0.845
"XGBoost ranker, unseen culprit service",0.944,0.993,1.000,0.986,0.970
"XGBoost, unseen system, metrics only",0.878,0.963,0.982,0.947,0.922
"XGBoost, unseen system, metrics + logs",0.874,0.963,0.978,0.947,0.921
"XGBoost, unseen system, metrics + logs + traces (all)",0.744,0.959,0.982,0.909,0.845
"XGBoost, unseen system, all, without within-case ranks",0.867,0.959,0.970,0.940,0.915
1D CNN (grouped CV),0.930,0.982,1.000,0.976,0.959


In [2]:
pd.DataFrame(ml["fault_type"]).T.round(3).rename_axis("fault-type classifier (6 classes)")

,accuracy,macro_F1
fault-type classifier (6 classes),,
service from the ranker (end-to-end),0.826,0.825
true service (oracle),0.848,0.847


## Conclusions
1. **Root-cause service:** a learned XGBoost ranker puts the true culprit first in **94.4%** of the 270 cases (top-3: 99.0%), vs 70.7% for the best rule, 51.1% for a BARO-style z-score and 5.5% for a random guess. The gain is significant (95% CI +17.8 to +28.9 points).
2. **Generalisation:**
   - A culprit service never seen as a culprit: 94.4%, no drop.
   - A completely new system: **87.8%** with metrics-only features, 85.9% with the CNN. Adding trace features drops this to 74.4%, because trace patterns learned from one system don't transfer.
3. **Fault type:** 82.6% accuracy over 6 fault types, end-to-end.
4. **Where it fails:** network delay and packet loss, where the symptom spreads to the callers.
5. **Deep learning:** a 1D CNN on raw time series matches XGBoost (93.0% vs 94.1%, not significant) without feature engineering, at much higher training cost.

**Comparison with IncidentLens** (ITSM ticket closure-category prediction, accuracy 0.718):
- RCAEval is the stronger project for a resume. Its labels are real (faults were injected, so the root cause is known), it has a published benchmark and baselines, and it was checked on unseen services and unseen systems.
- IncidentLens shows a different skill: messy business data and a served model with API and UI.